## AND-104 Task 6: Pre-Compute and Serve Predictions

Re-trains the best Module 3 model (HistGradientBoosting) on the full feature matrix,
scores every elevator using its most recent inspection row, and saves the results to
`data/predictions.csv` for the Go API to serve.

In [15]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier

BASE = Path('../data')

## 1. Load Data

In [16]:
df = pd.read_csv(BASE / 'feature_matrix.csv', parse_dates=['inspection_date'])

print(f'Rows: {len(df):,}')
print(f'Unique elevators: {df["ElevatingDevicesNumber"].nunique():,}')
print(f'Date range: {df["inspection_date"].min().date()} → {df["inspection_date"].max().date()}')

Rows: 141,789
Unique elevators: 40,916
Date range: 2011-01-04 → 2017-01-09


## 2. Feature Engineering

Replicates the three derived ratio features from Module 3 (`ml_pipeline.ipynb`). Raw counts
correlate with inspection history length; ratios generalise across elevators with different
amounts of history. `clip(lower=1)` avoids division-by-zero on first inspections.

In [17]:
df['pass_rate'] = (
    df['prior_outcome_counts_passed'] / df['prior_inspection_count'].clip(lower=1)
)
df['needs_action_rate'] = (
    df['prior_outcome_counts_needs_action'] / df['prior_inspection_count'].clip(lower=1)
)
df['orders_per_inspection'] = (
    df['prior_order_count'] / df['prior_inspection_count'].clip(lower=1)
)

bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print(f'Total features after engineering: {df.shape[1]}')

Total features after engineering: 38


## 3. Train Model on Full Dataset

Same model type and hyperparameters as Module 3. Training on all available datagives the model more signal before scoring the fleet.

In [18]:
NON_FEATURE_COLS = ['ElevatingDevicesNumber', 'InspectionNumber', 'inspection_date', 'location']
TARGET = 'outcome_binary'

X_all = df.drop(columns=NON_FEATURE_COLS + [TARGET])
y_all = df[TARGET]

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', HistGradientBoostingClassifier(
        max_iter=2000,
        learning_rate=0.02,
        max_depth=5,
        min_samples_leaf=20,
        n_iter_no_change=30,
        validation_fraction=0.1,
        random_state=42,
    )),
])

pipeline.fit(X_all, y_all)
print(f'Trained on {len(X_all):,} rows with {X_all.shape[1]} features')
print(f'Classes: {list(pipeline.classes_)}')

Trained on 141,789 rows with 33 features
Classes: ['Needs Action', 'Passed']


## 4. Score Most-Recent Row per Elevator

Each elevator is scored using its latest inspection row — the one that captures
the most complete and up-to-date feature state. Risk score = P(Needs Action).

In [19]:
latest = (
    df.sort_values('inspection_date')
      .groupby('ElevatingDevicesNumber')
      .last()
      .reset_index()
)

X_pred = latest.drop(columns=NON_FEATURE_COLS + [TARGET])

needs_action_idx = list(pipeline.classes_).index('Needs Action')
risk_scores = pipeline.predict_proba(X_pred)[:, needs_action_idx]

print(f'Scored {len(latest):,} elevators')
print(f'Score range: [{risk_scores.min():.4f}, {risk_scores.max():.4f}]')

Scored 40,916 elevators
Score range: [0.1078, 0.9924]


## 5. Assign Risk Levels and Validate

In [20]:
def assign_risk_level(score: float) -> str:
    if score >= 0.7:
        return 'high'
    if score >= 0.4:
        return 'medium'
    return 'low'

predictions = pd.DataFrame({
    'elevator_id': latest['ElevatingDevicesNumber'].values,
    'risk_score': risk_scores,
    'risk_level': [assign_risk_level(s) for s in risk_scores],
    'model_version': 'v1.0',
    'prediction_date': str(date.today()),
})

# --- Validation ---
all_elevators = df['ElevatingDevicesNumber'].unique()
covered = set(predictions['elevator_id'])
missing = set(all_elevators) - covered

assert len(missing) == 0, f'{len(missing)} elevators missing from predictions'
assert predictions['risk_score'].between(0, 1).all(), 'Scores outside [0, 1]'
assert predictions['risk_level'].nunique() > 1, 'All elevators in the same risk level'

level_counts = predictions['risk_level'].value_counts()

print('=== Prediction Summary ===')
print(f'Total elevators:  {len(predictions):,}')
print(f'High risk:        {level_counts.get("high", 0):,}')
print(f'Medium risk:      {level_counts.get("medium", 0):,}')
print(f'Low risk:         {level_counts.get("low", 0):,}')
print(f'Min score:        {predictions["risk_score"].min():.4f}')
print(f'Max score:        {predictions["risk_score"].max():.4f}')
print(f'Mean score:       {predictions["risk_score"].mean():.4f}')
print(f'Model version:    {predictions["model_version"].iloc[0]}')
print(f'Prediction date:  {predictions["prediction_date"].iloc[0]}')
print('\nAll validations passed.')

=== Prediction Summary ===
Total elevators:  40,916
High risk:        12,325
Medium risk:      26,588
Low risk:         2,003
Min score:        0.1078
Max score:        0.9924
Mean score:       0.6108
Model version:    v1.0
Prediction date:  2026-05-28

All validations passed.


## 6. Save Predictions

In [21]:
out_path = BASE / 'predictions.csv'
predictions.to_csv(out_path, index=False)
print(f'Saved {len(predictions):,} predictions to {out_path}')

Saved 40,916 predictions to ../data/predictions.csv
